# 02 - Titanic: classification and fare regression

Fit three survival classifiers using leakage-safe preprocessing, tune a random forest with out-of-bag scoring, evaluate a separate fare-regression model, and serialize the selected classification pipeline.

## 1. Imports and local data loading

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

sns.set_theme(style="whitegrid")

analytics_dir = Path.cwd()
if analytics_dir.name.lower() != "analytics":
    analytics_dir = analytics_dir / "analytics"
csv_path = analytics_dir / "titanic.csv"
if not csv_path.is_file():
    raise FileNotFoundError(f"Local Titanic dataset not found: {csv_path.resolve()}")

df = pd.read_csv(csv_path)
print(f"Data loaded successfully. Shape: {df.shape}")
df.head()

Data loaded successfully. Shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. Stratified split and leak-free preprocessing

The held-out test data is kept out of preprocessing fitting and hyperparameter selection. The column transformer only exposes the stated feature groups; all other source columns are dropped.

In [2]:
X = df.drop(columns=["survived"])
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ["age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked", "class", "who"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print(f"Training rows: {len(X_train)}; test rows: {len(X_test)}")
print("Training survival rate:", round(y_train.mean(), 3))
print("Test survival rate:", round(y_test.mean(), 3))

Training rows: 712; test rows: 179
Training survival rate: 0.383
Test survival rate: 0.385


## 3. Train and evaluate classifiers

All reported metrics and confusion matrices use the same stratified held-out test set.

In [3]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

evaluation_results = []
fitted_classifiers = {}

for name, clf in classifiers.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    fitted_classifiers[name] = pipe

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)

    evaluation_results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": roc_auc
    })

    print(f"--- {name} ---")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1-Score: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred, zero_division=0))
    print("-" * 60)

results_df = pd.DataFrame(evaluation_results).set_index("Model")
display(results_df.style.format("{:.4f}"))

--- Logistic Regression ---
Accuracy: 0.8212 | Precision: 0.8136 | Recall: 0.6957 | F1-Score: 0.7500 | ROC-AUC: 0.8664
Confusion Matrix:
 [[99 11]
 [21 48]]
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.90      0.86       110
           1       0.81      0.70      0.75        69

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179

------------------------------------------------------------
--- Decision Tree ---
Accuracy: 0.8268 | Precision: 0.8167 | Recall: 0.7101 | F1-Score: 0.7597 | ROC-AUC: 0.8543
Confusion Matrix:
 [[99 11]
 [20 49]]
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.90      0.86       110
           1       0.82      0.71      0.76        69

    accuracy                           0.83       179
   macro avg       0.82      0.81     

--- Random Forest ---
Accuracy: 0.8212 | Precision: 0.7937 | Recall: 0.7246 | F1-Score: 0.7576 | ROC-AUC: 0.8391
Confusion Matrix:
 [[97 13]
 [19 50]]
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.88      0.86       110
           1       0.79      0.72      0.76        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179

------------------------------------------------------------


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Model,,,,,
Logistic Regression,0.8212,0.8136,0.6957,0.7500,0.8664
Decision Tree,0.8268,0.8167,0.7101,0.7597,0.8543
Random Forest,0.8212,0.7937,0.7246,0.7576,0.8391


## 4. Random-forest tuning with GridSearchCV and OOB scoring

Grid search uses training data only. The test set remains reserved for the classifiers' held-out evaluation.

In [4]:
rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, oob_score=True))
])

param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [3, 5, 10],
    "classifier__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best Hyperparameters Found:")
print(grid_search.best_params_)
print(f"Best Cross-Validation F1 Score: {grid_search.best_score_:.4f}")
best_pipeline = grid_search.best_estimator_
print(f"Best estimator OOB score: {best_pipeline.named_steps['classifier'].oob_score_:.4f}")

Best Hyperparameters Found:
{'classifier__max_depth': 10, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}
Best Cross-Validation F1 Score: 0.7581
Best estimator OOB score: 0.8230


## 5. Regression side task: predict fare

The regression pipeline predicts fare without using `survived`. Adjusted R-squared uses the number of transformed predictor columns after preprocessing.

In [5]:
df_reg = df.dropna(subset=["fare"]).copy()
X_reg = df_reg.drop(columns=["fare", "survived"])
y_reg = df_reg["fare"]

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg_numeric = ["age", "sibsp", "parch"]
reg_categorical = ["sex", "embarked", "class", "who", "pclass"]

reg_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), reg_numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), reg_categorical)
    ],
    remainder="drop"
)

reg_pipeline = Pipeline(steps=[
    ("preprocessor", reg_preprocessor),
    ("regressor", Ridge(alpha=1.0))
])
reg_pipeline.fit(X_reg_train, y_reg_train)
y_reg_pred = reg_pipeline.predict(X_reg_test)

mae = mean_absolute_error(y_reg_test, y_reg_pred)
mse = mean_squared_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_reg_test, y_reg_pred)
n = len(y_reg_test)
p = reg_pipeline.named_steps["preprocessor"].transform(X_reg_test).shape[1]
if n <= p + 1:
    raise ValueError(f"Adjusted R-squared requires n > p + 1; received n={n}, p={p}.")
adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

print("Regression Evaluation on Fare:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R-squared Score: {r2:.4f}")
print(f"Adjusted R-squared: {adj_r2:.4f}")
print(f"Adjusted R-squared predictors (one-hot expanded): {p}")

Regression Evaluation on Fare:
MAE: 18.68
RMSE: 30.82
R-squared Score: 0.3860
Adjusted R-squared: 0.3211
Adjusted R-squared predictors (one-hot expanded): 17


## 6. Serialize and verify the tuned classifier

In [6]:
model_path = analytics_dir / "titanic_pipeline.joblib"
joblib.dump(best_pipeline, model_path)
print(f"Model successfully saved to {model_path}")

loaded_pipeline = joblib.load(model_path)
sample_prediction = loaded_pipeline.predict(X_test.head(1))
print("Sanity check prediction on test sample:", sample_prediction)
print("Matches original fitted pipeline:", np.array_equal(sample_prediction, best_pipeline.predict(X_test.head(1))))

Model successfully saved to C:\zepto-data-ai-platform\analytics\titanic_pipeline.joblib
Sanity check prediction on test sample: [0]
Matches original fitted pipeline: True
